In [3]:
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import os
import requests
PASSWORD = os.getenv('SNOWSQL_PWD')
print(PASSWORD)

5eWxWv4EvyCkhkY


In [11]:
try:
    ctx = snowflake.connector.connect(
        user='ABHINAVSHARMA2002',
        password=PASSWORD,
        account='qraojwa-yg67137'
    )
    cs = ctx.cursor()
    try:
        cs.execute("CREATE WAREHOUSE IF NOT EXISTS task_1_warehouse_mg")
        cs.execute("CREATE DATABASE IF NOT EXISTS testdb_mg")
        cs.execute("USE DATABASE testdb_mg")
        cs.execute("CREATE SCHEMA IF NOT EXISTS task_2_mg")
        cs.execute("USE WAREHOUSE task_1_warehouse_mg")
        cs.execute("USE SCHEMA task_2_mg")
        runQueries(cs)
        cs.execute('SELECT * FROM "product_profit_f" LIMIT 10')
        print(cs.fetchall())
    except Exception as e:
        print(f"Error: {e}") 
    finally:
        cs.close()
        ctx.close()
except Exception as e:
    print(f"Error: {e}")

[(6, 'Gaming Console', 225448.11000000002, 45183.06, 20.041445457227383), (18, 'E-Reader', 826174.8, 175288.66, 21.216897441074213), (1, 'Laptop', 846371.96, 162435.11000000002, 19.191929515245285), (20, 'Noise-Canceling Headphones', 185011.84, 36116.92, 19.52141008921375), (4, 'Smartwatch', 692624.64, 141134.54, 20.376771464555464), (17, 'Wireless Router', 515158.42000000004, 107449.71, 20.857605316826618), (9, 'External Hard Drive', 804172.89, 154604.04, 19.225224068421408), (14, 'Action Camera', 805162.68, 163266.4, 20.277442566016596), (2, 'Smartphone', 1202853.36, 249768.37, 20.764656632791876), (3, 'Tablet', 123973.1, 25195.3, 20.323199145621103)]


In [1]:
def runQueries(cs):    
    query = """
    CREATE OR REPLACE VIEW "product_profit_f" AS 
    SELECT 
    p."product_id", 
    p."product_name", 
    SUM(o."total_price") AS "total_revenue",
    SUM(o."profit") AS "total_profit",
    CASE 
        WHEN SUM(o."total_price") > 0 
        THEN (SUM(o."profit") / SUM(o."total_price")) * 100
        ELSE 0 
    END AS "profit_margin_percentage"
    FROM products p
    JOIN orders o 
    ON p."product_id" = o."product_id" 
    GROUP BY p."product_id", p."product_name";
    """
    cs.execute(query)